Chapter 4 introduces Cohort Analysis. This is where the book transitions from basic data manipulation to genuine analytical modeling in SQL.

In business analytics and risk modeling, a cohort is a group of entities (customers, borrowers, accounts, policyholders, or assets) that share a common starting event within a specific time window (e.g., accounts opened in Q1 2024, or policies underwritten in March 2025).

Instead of measuring static cross-sectional snapshots, cohort analysis tracks behavior across age/vintage over time.


| Concept              | Problem Solved                                                  | Primary SQL Technique                                          |
| :------------------- | :-------------------------------------------------------------- | :------------------------------------------------------------- |
| **Cohort Anchoring** | Permanently tagging entities by origin date                     | `MIN(event_date) OVER (PARTITION BY entity_id)` or CTE         |
| **Elapsed Indexing** | Normalizing calendar dates to relative vintage age ($t_0, t_1$) | `DATEDIFF('month', cohort_date, activity_date)`                |
| **Retention Rates**  | Tracking survival rates relative to initial population size     | `active_entities_at_t / total_cohort_size_t0`                  |
| **Matrix Pivoting**  | Converting long temporal logs into wide triangular matrices     | Conditional Aggregation (`SUM(CASE WHEN period=N THEN 1 END)`) |


In [ ]:
-- 1. Defining the Cohort Base (First-Event Anchor)

-- To analyze a cohort, every entity must be assigned a permanent "cohort date"—typically their earliest timestamp in the database.

-- The Pattern: Compute each entity’s minimum event date using a CTE or subquery:

-- SQL

SELECT user_id, MIN(event_date) AS cohort_date
FROM activity_logs
GROUP BY user_id

-- Why it matters: Once established, this anchor date remains fixed regardless of future activity.

In [ ]:
-- cohort analysis example

-- SQL to create a cohort activity table
CREATE TABLE cohort_activity AS
SELECT
    a.user_id,
    c.cohort_date,
    a.event_date,
    DATEDIFF('month', c.cohort_date, a.event_date) AS period_number
FROM activity_logs a
JOIN (
    SELECT user_id, MIN(event_date) AS cohort_date
    FROM activity_logs
    GROUP BY user_id
) c
ON a.user_id = c.user_id;

In [ ]:
-- 2. Time-Lapse Indexing (Periods Since Inception / Vintage Age)

-- Instead of plotting calendar time on the x-axis (e.g., Jan 2025, Feb 2025), cohort analysis plots elapsed periods since origin ($t_0, t_1, t_2, \dots$).

-- Calculation: AGE_IN_MONTHS = (event_date - cohort_date).In SQL: Using DuckDB's date arithmetic or date truncation:

-- SQL

DATEDIFF('month', cohort_month, activity_month) AS period_number;

In [ ]:
-- 3. Retention, Churn, and Survival Curves

-- Once activities are indexed by period_number, you aggregate active counts at each period and divide by the initial size of the cohort at $t_0$.

-- Risk/Credit Analogy: This is identical to calculating cumulative default rates or loss severity curves across loan underwriting vintages.

$$\text{Retention}_{c, t} = \frac{\text{Active Entities in Cohort } c \text{ at Period } t}{\text{Total Starting Entities in Cohort } c \text{ at Period } 0}$$

In [ ]:
-- 4. Cross-Tabulation & Matrix Pivot Representation

-- Cohort analysis is almost always visualized as a triangular matrix (Rows = Cohort Start Month, Columns = Period $t_0, t_1, t_2, \dots$).

-- SQL Pattern: Conditional aggregation using CASE WHEN or DuckDB's native PIVOT clause to transform dynamic rows into fixed period columns.

-- Example using CASE WHEN
SELECT
    cohort_month,
    SUM(CASE WHEN period_number = 0 THEN 1 ELSE 0 END) AS period_0,
    SUM(CASE WHEN period_number = 1 THEN 1 ELSE 0 END) AS period_1,
    SUM(CASE WHEN period_number = 2 THEN 1 ELSE 0 END) AS period_2
FROM cohort_activity
GROUP BY cohort_month
ORDER BY cohort_month;

In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect(database=":memory:")

-- 1. Generate Synthetic Event Logs (Users arriving across 2026 and returning over time)
conn.execute(
CREATE TABLE user_events AS
SELECT
    'user_' || (random() * 50)::INT AS user_id,
    TIMESTAMP '2026-01-01' + INTERVAL (random() * 180)::DAY AS event_time
FROM range(1000);
)

-- # 2. Chapter 4 Cohort Analysis Pipeline: Anchor -> Index -> Aggregate -> Pivot Matrix
cohort_matrix_df = conn.execute(
WITH user_first_activity AS (
    -- Step 1: Establish the Fixed Cohort Date for each user
    SELECT
        user_id,
        DATE_TRUNC('month', MIN(event_time)) AS cohort_month
    FROM user_events
    GROUP BY user_id
),
user_monthly_activity AS (
    -- Step 2: Extract distinct active months per user
    SELECT DISTINCT
        user_id,
        DATE_TRUNC('month', event_time) AS activity_month
    FROM user_events
),
cohort_size AS (
    -- Step 3: Count total starting users per cohort (t0 baseline)
    SELECT
        cohort_month,
        COUNT(DISTINCT user_id) AS starting_users
    FROM user_first_activity
    GROUP BY cohort_month
),
cohort_activity AS (
    -- Step 4: Join activity back to cohort date and calculate elapsed period index
    SELECT
        f.cohort_month,
        DATEDIFF('month', f.cohort_month, a.activity_month) AS period_number,
        COUNT(DISTINCT a.user_id) AS active_users
    FROM user_first_activity f
    JOIN user_monthly_activity a ON f.user_id = a.user_id
    GROUP BY 1, 2
)
-- Step 5: Render Triangular Retention Matrix (% of starting cohort)
SELECT
    ca.cohort_month::DATE AS cohort,
    cs.starting_users AS size_t0,
    ROUND(MAX(CASE WHEN period_number = 0 THEN active_users END) * 100.0 / cs.starting_users, 1) AS "m0_%",
    ROUND(MAX(CASE WHEN period_number = 1 THEN active_users END) * 100.0 / cs.starting_users, 1) AS "m1_%",
    ROUND(MAX(CASE WHEN period_number = 2 THEN active_users END) * 100.0 / cs.starting_users, 1) AS "m2_%",
    ROUND(MAX(CASE WHEN period_number = 3 THEN active_users END) * 100.0 / cs.starting_users, 1) AS "m3_%"
FROM cohort_activity ca
JOIN cohort_size cs ON ca.cohort_month = cs.cohort_month
GROUP BY ca.cohort_month, cs.starting_users
ORDER BY ca.cohort_month;
).df()

cohort_matrix_df